In [ ]:
# Imports et paramètres globaux
import time, platform, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from scipy.stats import wilcoxon
from sklearn.datasets import load_breast_cancer, fetch_openml
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, GridSearchCV)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")          # masquer les avertissements

R      = 10      # répétitions
P      = 20      # taille de population
T      = 30      # itérations
ALPHA  = 0.99    # poids de l'erreur
K_NN   = 5       # voisins du k-NN
TEST   = 0.30    # proportion de test

os.makedirs("figures", exist_ok=True)
print(f"Python {platform.python_version()} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")
print(f"Budget par exécution : {P} + {P}×{T} = {P + P*T} évaluations de la fitness")

## Jeux de données

In [ ]:
# Chargement des jeux de données
UCI = {
    "sonar": "https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data",
    "ionosphere": "https://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data",
}

def load_one(name):
    try:
        X, y = fetch_openml(name, version=1, return_X_y=True, as_frame=False)
    except Exception:
        df = pd.read_csv(UCI[name], header=None)
        X, y = df.iloc[:, :-1].values, df.iloc[:, -1].values
    return X.astype(float), LabelEncoder().fit_transform(y)

def load_datasets():
    data = {}
    X, y = load_breast_cancer(return_X_y=True)
    data["Breast Cancer"] = (X, y)
    data["Sonar"] = load_one("sonar")
    data["Ionosphere"] = load_one("ionosphere")
    return data

data = load_datasets()
for name, (X, y) in data.items():
    print(f"{name:14s} n = {X.shape[0]:4d}   N = {X.shape[1]:3d}   effectifs des classes = {np.bincount(y)}")

## Protocole et fitness

In [ ]:
# Découpage, fitness, évaluation finale
def split(X, y, seed):
    # découpage 70/30 + standardisation (train)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST, stratify=y,
                                              random_state=seed)
    sc = StandardScaler().fit(X_tr)
    return sc.transform(X_tr), sc.transform(X_te), y_tr, y_te


class Fitness:
    # fitness wrapper, à minimiser
    def __init__(self, X_tr, y_tr, seed, alpha=ALPHA):
        self.X, self.y, self.alpha = X_tr, y_tr, alpha
        self.N = X_tr.shape[1]
        self.cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        self.cache = {}      # cache des masques
        self.calls = 0       # compteur d'évaluations

    def __call__(self, mask):
        self.calls += 1
        key = mask.tobytes()
        if key not in self.cache:
            m = mask.astype(bool)
            if m.sum() == 0:
                self.cache[key] = 1.0
            else:
                acc = cross_val_score(KNeighborsClassifier(K_NN), self.X[:, m], self.y,
                                      cv=self.cv).mean()
                self.cache[key] = self.alpha * (1 - acc) + (1 - self.alpha) * m.sum() / self.N
        return self.cache[key]


def test_accuracy(mask, X_tr, X_te, y_tr, y_te):
    # précision sur le test
    m = mask.astype(bool)
    if m.sum() == 0:                          # aucune variable : classe majoritaire
        return np.mean(y_te == np.bincount(y_tr).argmax())
    clf = KNeighborsClassifier(K_NN).fit(X_tr[:, m], y_tr)
    return clf.score(X_te[:, m], y_te)

## Références simples

In [ ]:
# Références
def select_toutes(X_tr, y_tr, seed):
    return np.ones(X_tr.shape[1], dtype=np.int8)

def select_filtre(X_tr, y_tr, seed):
    N = X_tr.shape[1]
    ks = sorted({max(1, int(N * p)) for p in (0.10, 0.20, 0.30, 0.50, 0.75)})
    pipe = Pipeline([("sel", SelectKBest(f_classif)), ("knn", KNeighborsClassifier(K_NN))])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    gs = GridSearchCV(pipe, {"sel__k": ks}, cv=cv).fit(X_tr, y_tr)
    return gs.best_estimator_.named_steps["sel"].get_support().astype(np.int8)

## Algorithme génétique

In [ ]:
# Algorithme génétique
def run_ga(fit, N, rng, P=P, T=T, pc=0.9, pm=None, tour=3):
    pm = 1.0 / N if pm is None else pm
    pop = (rng.random((P, N)) < 0.5).astype(np.int8)          # initialisation
    fits = np.array([fit(ind) for ind in pop])
    b = fits.argmin()
    best, best_f = pop[b].copy(), fits[b]
    curve = [best_f]

    def tournoi():
        idx = rng.choice(P, size=tour, replace=False)          # tournoi
        return pop[idx[fits[idx].argmin()]]                    # meilleur candidat

    for t in range(T):
        enfants = [best.copy()]                                # élitisme
        while len(enfants) < P:
            p1, p2 = tournoi(), tournoi()
            if rng.random() < pc:                              # croisement uniforme
                masque = rng.random(N) < 0.5
                e1, e2 = np.where(masque, p1, p2), np.where(masque, p2, p1)
            else:
                e1, e2 = p1.copy(), p2.copy()
            for e in (e1, e2):                                 # mutation bit à bit
                flip = rng.random(N) < pm
                e[flip] = 1 - e[flip]
                enfants.append(e)
        pop = np.array(enfants[:P], dtype=np.int8)
        fits = np.array([fit(ind) for ind in pop])
        b = fits.argmin()
        if fits[b] < best_f:
            best, best_f = pop[b].copy(), fits[b]
        curve.append(best_f)
    return best, np.array(curve)

## PSO binaire

In [ ]:
# PSO binaire
def run_bpso(fit, N, rng, P=P, T=T, w_max=0.9, w_min=0.4, c1=2.0, c2=2.0, vmax=6.0):
    X = (rng.random((P, N)) < 0.5).astype(np.int8)              # positions
    V = rng.uniform(-1, 1, size=(P, N))                         # vitesses
    fits = np.array([fit(x) for x in X])
    pbest, pbest_f = X.copy(), fits.copy()                      # pbest
    g = pbest_f.argmin()
    gbest, gbest_f = pbest[g].copy(), pbest_f[g]                # gbest
    curve = [gbest_f]

    for t in range(T):
        w = w_max - (w_max - w_min) * t / max(T - 1, 1)         # inertie décroissante
        r1, r2 = rng.random((P, N)), rng.random((P, N))
        V = w * V + c1 * r1 * (pbest - X) + c2 * r2 * (gbest - X)
        V = np.clip(V, -vmax, vmax)
        proba = 1.0 / (1.0 + np.exp(-V))                        # sigmoïde
        X = (rng.random((P, N)) < proba).astype(np.int8)
        fits = np.array([fit(x) for x in X])
        mieux = fits < pbest_f
        pbest[mieux], pbest_f[mieux] = X[mieux], fits[mieux]
        g = pbest_f.argmin()
        if pbest_f[g] < gbest_f:
            gbest, gbest_f = pbest[g].copy(), pbest_f[g]
        curve.append(gbest_f)
    return gbest, np.array(curve)

## Test rapide

In [ ]:
# Test rapide
X, y = data["Breast Cancer"]
X_tr, X_te, y_tr, y_te = split(X, y, seed=0)
for nom, algo in [("AG", run_ga), ("BPSO", run_bpso)]:
    fit = Fitness(X_tr, y_tr, seed=0)
    t0 = time.perf_counter()
    mask, curve = algo(fit, X_tr.shape[1], np.random.default_rng(0))
    dt = time.perf_counter() - t0
    print(f"{nom:5s} |S| = {mask.sum():2d}   acc test = {test_accuracy(mask, X_tr, X_te, y_tr, y_te):.3f}"
          f"   fitness = {curve[-1]:.4f}   appels = {fit.calls}   temps = {dt:.1f} s")

## Expériences

In [ ]:
# Boucle d'expériences
METHODES = ["Toutes", "Filtre", "AG", "BPSO"]
rows, curves = [], {}
t_global = time.perf_counter()

for ds, (X, y) in data.items():
    for r in range(R):
        X_tr, X_te, y_tr, y_te = split(X, y, seed=r)
        N = X_tr.shape[1]
        for meth in METHODES:
            t0 = time.perf_counter()
            if meth == "Toutes":
                mask = select_toutes(X_tr, y_tr, r)
            elif meth == "Filtre":
                mask = select_filtre(X_tr, y_tr, r)
            else:
                fit = Fitness(X_tr, y_tr, seed=r)
                algo = run_ga if meth == "AG" else run_bpso
                # graine fixe par méthode
                rng = np.random.default_rng(1000 * r + (1 if meth == "AG" else 2))
                mask, curve = algo(fit, N, rng)
                curves[(ds, meth, r)] = curve
            dt = time.perf_counter() - t0
            rows.append(dict(dataset=ds, methode=meth, rep=r,
                             acc=test_accuracy(mask, X_tr, X_te, y_tr, y_te),
                             nsel=int(mask.sum()), temps=dt,
                             masque="".join(map(str, mask))))
        print(f"{ds:14s} répétition {r+1:2d}/{R} terminée  ({time.perf_counter() - t_global:6.0f} s)")

res = pd.DataFrame(rows)
res.to_csv("resultats_bruts.csv", index=False)
print("\nTerminé. Résultats bruts : resultats_bruts.csv")

## Résumé

In [ ]:
# Résumé
DATASETS = list(data.keys())
resume = res.groupby(["dataset", "methode"]).agg(
    acc_moy=("acc", "mean"), acc_std=("acc", "std"),
    nsel_moy=("nsel", "mean"), nsel_std=("nsel", "std"),
    temps_moy=("temps", "mean")).reindex(
    pd.MultiIndex.from_product([DATASETS, METHODES]))
resume.index.names = ["dataset", "methode"]
resume.to_csv("resume.csv")

pd.options.display.float_format = "{:.4f}".format
resume

## Test de Wilcoxon

In [ ]:
# Wilcoxon
def p_wilcoxon(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if np.allclose(a, b):
        return 1.0                       # séries identiques
    try:
        return wilcoxon(a, b).pvalue
    except ValueError:
        return 1.0

def serie(ds, meth, col):
    return res[(res.dataset == ds) & (res.methode == meth)].sort_values("rep")[col].values

tests = []
for ds in DATASETS:
    tests.append(dict(dataset=ds,
        p_acc_AG_BPSO=p_wilcoxon(serie(ds, "AG", "acc"), serie(ds, "BPSO", "acc")),
        p_nsel_AG_BPSO=p_wilcoxon(serie(ds, "AG", "nsel"), serie(ds, "BPSO", "nsel")),
        p_acc_AG_Filtre=p_wilcoxon(serie(ds, "AG", "acc"), serie(ds, "Filtre", "acc")),
        p_acc_BPSO_Filtre=p_wilcoxon(serie(ds, "BPSO", "acc"), serie(ds, "Filtre", "acc")),
        p_acc_AG_Toutes=p_wilcoxon(serie(ds, "AG", "acc"), serie(ds, "Toutes", "acc")),
        p_acc_BPSO_Toutes=p_wilcoxon(serie(ds, "BPSO", "acc"), serie(ds, "Toutes", "acc"))))
tests = pd.DataFrame(tests).set_index("dataset")
tests.to_csv("wilcoxon.csv")
tests

## Figures

In [ ]:
# Figures
plt.rcParams.update({"font.size": 10})
COUL = {"AG": "#1f77b4", "BPSO": "#d62728"}

# convergence
fig, axes = plt.subplots(1, len(DATASETS), figsize=(13, 3.6))
for ax, ds in zip(axes, DATASETS):
    for meth in ["AG", "BPSO"]:
        C = np.array([curves[(ds, meth, r)] for r in range(R)])
        m, s = C.mean(0), C.std(0)
        it = np.arange(len(m))
        ax.plot(it, m, label=meth, color=COUL[meth])
        ax.fill_between(it, m - s, m + s, color=COUL[meth], alpha=0.15)
    ax.set_title(ds); ax.set_xlabel("Itération"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Meilleure fitness (à minimiser)")
axes[0].legend()
plt.tight_layout(); plt.savefig("figures/convergence.png", dpi=200); plt.show()

# boxplot des précisions
fig, axes = plt.subplots(1, len(DATASETS), figsize=(13, 3.6))
for ax, ds in zip(axes, DATASETS):
    ax.boxplot([100 * serie(ds, m, "acc") for m in METHODES])
    ax.set_xticks(range(1, len(METHODES) + 1), METHODES)
    ax.set_title(ds); ax.grid(alpha=0.3)
axes[0].set_ylabel("Précision sur le test (%)")
plt.tight_layout(); plt.savefig("figures/boxplot_precision.png", dpi=200); plt.show()

## Export LaTeX

In [ ]:
# Génération de resultats_auto.tex
NOM = {"Toutes": "la référence Toutes", "Filtre": "le filtre", "AG": "l'AG", "BPSO": "le BPSO"}
NV = {ds: data[ds][0].shape[1] for ds in DATASETS}
DS_MAX = max(DATASETS, key=lambda d: NV[d])           # jeu le plus grand

def num(x, d=1):                                       # format français
    return f"{x:.{d}f}".replace(".", "{,}")
def pct(x, d=1):
    return num(100 * x, d)
def pts(x):                                            # écart en points
    v = 100 * x
    return f"${'+' if v >= 0 else '-'}{num(abs(v))}$~point{'s' if abs(v) >= 2 else ''}"
def pval(p):
    return "$p < 0{,}001$" if p < 0.001 else f"$p = {num(p, 3)}$"
def M(ds, m, col):
    return resume.loc[(ds, m), col]
def P(ds, col):
    return tests.loc[ds, col]
def est_sig(p):
    return "est significative" if p < 0.05 else "n'est pas significative"
def liste(xs):
    xs = list(xs)
    return xs[0] if len(xs) == 1 else ", ".join(xs[:-1]) + " et " + xs[-1]
def macro(nom, corps):
    return f"\\newcommand{{\\{nom}}}{{%\n{corps}%\n}}\n\n"

def fmt(m, s, d=1, gras=False):
    t = f"{num(m, d)} \\pm {num(s, d)}"
    return f"$\\mathbf{{{t}}}$" if gras else f"${t}$"
def fmt_p(p):
    t = "< 0{,}001" if p < 0.001 else num(p, 3)
    return f"$\\mathbf{{{t}}}$" if p < 0.05 else f"${t}$"

out = [f"% Fichier généré automatiquement par le notebook le {time.strftime('%d/%m/%Y %H:%M')}\n",
       macro("NbRep", str(R))]

# tableaux
L = [r"\begin{table}[H]", r"\centering",
     rf"\caption{{Précision sur le test en \% (moyenne $\pm$ écart-type sur $R = {R}$ répétitions). Meilleure moyenne en gras.}}",
     r"\label{tab:acc}", r"\begin{tabular}{@{}lcccc@{}}", r"\toprule",
     r"Jeu de données & Toutes & Filtre & AG & BPSO \\", r"\midrule"]
for ds in DATASETS:
    moy = {m: M(ds, m, "acc_moy") for m in METHODES}
    best = max(moy.values())
    L.append(f"{ds} & " + " & ".join(fmt(100 * moy[m], 100 * M(ds, m, "acc_std"), 1,
                                          np.isclose(moy[m], best)) for m in METHODES) + r" \\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
out.append(macro("TabAcc", "\n".join(L)))

L = [r"\begin{table}[H]", r"\centering",
     r"\caption{Nombre de variables sélectionnées $|S|$ (moyenne $\pm$ écart-type). Plus petite moyenne en gras (hors Toutes).}",
     r"\label{tab:nvar}", r"\begin{tabular}{@{}lcccc@{}}", r"\toprule",
     r"Jeu de données & Toutes & Filtre & AG & BPSO \\", r"\midrule"]
for ds in DATASETS:
    moy = {m: M(ds, m, "nsel_moy") for m in METHODES}
    best = min(moy[m] for m in METHODES[1:])
    L.append(f"{ds} & ${NV[ds]}$ & " + " & ".join(fmt(moy[m], M(ds, m, "nsel_std"), 1,
                                                       np.isclose(moy[m], best)) for m in METHODES[1:]) + r" \\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
out.append(macro("TabNvar", "\n".join(L)))

L = [r"\begin{table}[H]", r"\centering",
     r"\caption{Temps moyen de sélection par exécution (secondes) et rapport au temps du filtre.}",
     r"\label{tab:temps}", r"\begin{tabular}{@{}lccccc@{}}", r"\toprule",
     r"Jeu de données & Filtre & AG & BPSO & AG / Filtre & BPSO / Filtre \\", r"\midrule"]
for ds in DATASETS:
    tf, ta, tb = (M(ds, m, "temps_moy") for m in ["Filtre", "AG", "BPSO"])
    L.append(f"{ds} & ${num(tf, 2)}$ & ${num(ta, 2)}$ & ${num(tb, 2)}$ & "
             f"$\\times {num(ta / tf, 0)}$ & $\\times {num(tb / tf, 0)}$ \\\\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
out.append(macro("TabTemps", "\n".join(L)))

L = [r"\begin{table}[H]", r"\centering",
     r"\caption{Test de Wilcoxon apparié, bilatéral (valeurs $p$ ; en gras si $p < 0{,}05$).}",
     r"\label{tab:wilcoxon}", r"\small", r"\begin{tabular}{@{}lcccccc@{}}", r"\toprule",
     r" & \multicolumn{2}{c}{AG vs BPSO} & AG vs Filtre & BPSO vs Filtre & AG vs Toutes & BPSO vs Toutes \\",
     r"\cmidrule(lr){2-3}\cmidrule(lr){4-4}\cmidrule(lr){5-5}\cmidrule(lr){6-6}\cmidrule(lr){7-7}",
     r"Jeu de données & précision & $|S|$ & précision & précision & précision & précision \\", r"\midrule"]
for ds in DATASETS:
    L.append(f"{ds} & " + " & ".join(fmt_p(P(ds, c)) for c in
             ["p_acc_AG_BPSO", "p_nsel_AG_BPSO", "p_acc_AG_Filtre", "p_acc_BPSO_Filtre",
              "p_acc_AG_Toutes", "p_acc_BPSO_Toutes"]) + r" \\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
out.append(macro("TabWilcoxon", "\n".join(L)))

out.append(macro("FigConvergence", "\n".join([
    r"\begin{figure}[H]", r"\centering",
    r"\includegraphics[width=\linewidth]{figures/convergence.png}",
    rf"\caption{{Convergence : meilleure fitness moyenne par itération ($\pm$ écart-type, {R} répétitions).}}",
    r"\label{fig:convergence}", r"\end{figure}"])))
out.append(macro("FigBoxplot", "\n".join([
    r"\begin{figure}[H]", r"\centering",
    r"\includegraphics[width=\linewidth]{figures/boxplot_precision.png}",
    rf"\caption{{Distribution de la précision sur le test sur les {R} répétitions.}}",
    r"\label{fig:boxplot}", r"\end{figure}"])))

# précision
T = []
for ds in DATASETS:
    a = {m: M(ds, m, "acc_moy") for m in METHODES}
    best = max(a, key=a.get)
    T.append(f"Sur {ds}, la meilleure précision moyenne est obtenue par {NOM[best]} ({pct(a[best])}~\\%). "
             f"Par rapport à l'utilisation de toutes les variables ({pct(a['Toutes'])}~\\%), l'écart est de "
             f"{pts(a['AG'] - a['Toutes'])} pour l'AG, {pts(a['BPSO'] - a['Toutes'])} pour le BPSO "
             f"et {pts(a['Filtre'] - a['Toutes'])} pour le filtre.")
stds = [100 * M(ds, m, "acc_std") for ds in DATASETS for m in METHODES]
T.append(f"Les écarts-types de la précision varient de {num(min(stds))} à {num(max(stds))}~points selon "
         f"la méthode et le jeu de données (tableau~\\ref{{tab:acc}}, figure~\\ref{{fig:boxplot}}).")
out.append(macro("ResPrecision", "\n".join(T)))

# variables
T = []
for ds in DATASETS:
    N = NV[ds]
    n = {m: M(ds, m, "nsel_moy") for m in METHODES}
    T.append(f"Sur {ds} ($N = {N}$), l'AG retient en moyenne {num(n['AG'])} variables "
             f"({num(100 * n['AG'] / N, 0)}~\\% de $N$), le BPSO {num(n['BPSO'])} "
             f"({num(100 * n['BPSO'] / N, 0)}~\\%) et le filtre {num(n['Filtre'])} "
             f"({num(100 * n['Filtre'] / N, 0)}~\\%).")
sd = {m: np.mean([M(ds, m, "nsel_std") for ds in DATASETS]) for m in ["Filtre", "AG", "BPSO"]}
T.append(f"L'écart-type moyen de $|S|$ entre répétitions est de {num(sd['AG'])} pour l'AG, "
         f"{num(sd['BPSO'])} pour le BPSO et {num(sd['Filtre'])} pour le filtre (tableau~\\ref{{tab:nvar}}).")
out.append(macro("ResVariables", "\n".join(T)))

# temps
rA = {ds: M(ds, "AG", "temps_moy") / M(ds, "Filtre", "temps_moy") for ds in DATASETS}
rB = {ds: M(ds, "BPSO", "temps_moy") / M(ds, "Filtre", "temps_moy") for ds in DATASETS}
tF = [M(ds, "Filtre", "temps_moy") for ds in DATASETS]
bpso_lent = [ds for ds in DATASETS if M(ds, "BPSO", "temps_moy") > M(ds, "AG", "temps_moy")]
T = [f"Le filtre sélectionne les variables en {num(min(tF), 2)} à {num(max(tF), 2)}~s selon le jeu de "
     f"données. La sélection par AG est {num(min(rA.values()), 0)} à {num(max(rA.values()), 0)} fois plus "
     f"lente, et celle par BPSO {num(min(rB.values()), 0)} à {num(max(rB.values()), 0)} fois plus lente "
     f"(tableau~\\ref{{tab:temps}})."]
if len(bpso_lent) == len(DATASETS):
    T.append("À budget égal, le BPSO est plus lent que l'AG sur les trois jeux de données.")
elif not bpso_lent:
    T.append("À budget égal, l'AG est plus lent que le BPSO sur les trois jeux de données.")
else:
    T.append(f"À budget égal, le BPSO est plus lent que l'AG sur {liste(bpso_lent)}, et plus rapide ailleurs.")
T.append("Le nombre d'évaluations étant identique, cet écart provient surtout du nombre de sous-ensembles "
         "distincts réellement évalués (la mémoïsation évite de réentraîner un masque déjà vu) et de la "
         "taille des sous-ensembles évalués.")
out.append(macro("ResTemps", "\n".join(T)))

# significativité
T = []
for ds in DATASETS:
    T.append(f"Sur {ds}, la différence de précision entre AG et BPSO {est_sig(P(ds, 'p_acc_AG_BPSO'))} "
             f"({pval(P(ds, 'p_acc_AG_BPSO'))}), et celle du nombre de variables "
             f"{est_sig(P(ds, 'p_nsel_AG_BPSO'))} ({pval(P(ds, 'p_nsel_AG_BPSO'))}). Par rapport au filtre, "
             f"la différence de précision de l'AG {est_sig(P(ds, 'p_acc_AG_Filtre'))} "
             f"({pval(P(ds, 'p_acc_AG_Filtre'))}) ; celle du BPSO {est_sig(P(ds, 'p_acc_BPSO_Filtre'))} "
             f"({pval(P(ds, 'p_acc_BPSO_Filtre'))}).")
T.append(f"Avec $R = {R}$ répétitions, la plus petite valeur $p$ atteignable par le test de Wilcoxon "
         f"bilatéral exact est d'environ ${num(2 / 2 ** R, 3)}$ (tableau~\\ref{{tab:wilcoxon}}).")
out.append(macro("ResSignif", "\n".join(T)))

# convergence
T, encore = [], {"AG": [], "BPSO": []}
for ds in DATASETS:
    fin, it95 = {}, {}
    for m in ["AG", "BPSO"]:
        c = np.array([curves[(ds, m, r)] for r in range(R)]).mean(0)
        fin[m] = c[-1]
        gain = c[0] - c[-1]
        it95[m] = int(np.argmax(c[0] - c >= 0.95 * gain)) if gain > 0 else 0
        if c[-6] - c[-1] > 1e-9:
            encore[m].append(ds)
    meilleur = "l'AG" if fin["AG"] < fin["BPSO"] else "le BPSO"
    T.append(f"Sur {ds}, {meilleur} atteint la meilleure fitness moyenne finale (AG : {num(fin['AG'], 4)} ; "
             f"BPSO : {num(fin['BPSO'], 4)}). L'AG réalise 95~\\% de son amélioration totale à l'itération "
             f"{it95['AG']}, le BPSO à l'itération {it95['BPSO']}.")
for m, nm in [("AG", "de l'AG"), ("BPSO", "du BPSO")]:
    if encore[m]:
        T.append(f"La courbe moyenne {nm} décroît encore au cours des cinq dernières itérations sur "
                 f"{liste(encore[m])}, ce qui suggère qu'un budget plus élevé pourrait encore l'améliorer.")
    else:
        T.append(f"La courbe moyenne {nm} est stabilisée au cours des cinq dernières itérations sur les trois jeux de données.")
T.append("(Figure~\\ref{fig:convergence}.)")
out.append(macro("ResConvergence", "\n".join(T)))

# hypothèses
def verdict(ok):
    k = sum(ok.values())
    if k == len(ok):
        return "confirmée", "sur les trois jeux de données"
    if k == 0:
        return "infirmée", "sur aucun jeu de données"
    return "partiellement confirmée", f"sur {liste([d for d in ok if ok[d]])} seulement"

# H1
ok1 = {}
for ds in DATASETS:
    ok_acc = all(M(ds, m, "acc_moy") >= M(ds, "Toutes", "acc_moy") or P(ds, f"p_acc_{m}_Toutes") >= 0.05
                 for m in ["AG", "BPSO"])
    ok_nsel = all(M(ds, m, "nsel_moy") <= NV[ds] / 2 for m in ["AG", "BPSO"])
    ok1[ds] = ok_acc and ok_nsel
v1, o1 = verdict(ok1)
H1 = (f"\\textbf{{H1 est {v1}}} {o1}. Critère : sur un jeu de données, H1 est vérifiée si la précision de "
      f"l'AG et du BPSO n'est pas significativement inférieure à celle de la référence Toutes et si chacun "
      f"retient au plus la moitié des variables (tableaux~\\ref{{tab:acc}}, \\ref{{tab:nvar}} et \\ref{{tab:wilcoxon}}).")

# H2
gain = {ds: max(M(ds, "AG", "acc_moy"), M(ds, "BPSO", "acc_moy")) - M(ds, "Filtre", "acc_moy") for ds in DATASETS}
plus_grand = max(gain, key=gain.get)
sig2 = any(min(P(ds, "p_acc_AG_Filtre"), P(ds, "p_acc_BPSO_Filtre")) < 0.05 for ds in DATASETS)
v2 = ("confirmée" if sig2 else "confirmée au niveau des moyennes, mais pas statistiquement") \
     if plus_grand == DS_MAX and gain[DS_MAX] > 0 else "infirmée"
H2 = (f"\\textbf{{H2 est {v2}}}. L'écart de précision entre la meilleure métaheuristique et le filtre vaut "
      + ", ".join(f"{pts(gain[ds])} sur {ds}" for ds in DATASETS)
      + f" ; il est le plus grand sur {plus_grand}"
      + (f", qui est bien le jeu de plus grande dimension ($N = {NV[DS_MAX]}$)." if plus_grand == DS_MAX
         else f", et non sur {DS_MAX} ($N = {NV[DS_MAX]}$).")
      + (" Cet écart est toutefois négatif sur ce jeu : le filtre y reste plus précis." if gain[plus_grand] <= 0 else "")
      + ("" if sig2 else " Aucune de ces différences n'est toutefois significative (tableau~\\ref{tab:wilcoxon})."))

# H3
rmin = min(min(rA.values()), min(rB.values()))
v3 = "confirmée" if rmin >= 10 else "infirmée"
H3 = (f"\\textbf{{H3 est {v3}}}. Le plus petit rapport de temps observé entre une métaheuristique et le "
      f"filtre est de {num(rmin, 0)} (tableau~\\ref{{tab:temps}}) ; le seuil d'un ordre de grandeur correspond "
      f"à un rapport de 10.")

# H4
bpso_sup = [ds for ds in DATASETS if M(ds, "BPSO", "acc_moy") > M(ds, "AG", "acc_moy") and P(ds, "p_acc_AG_BPSO") < 0.05]
ag_sup = [ds for ds in DATASETS if M(ds, "AG", "acc_moy") > M(ds, "BPSO", "acc_moy") and P(ds, "p_acc_AG_BPSO") < 0.05]
bpso_moy_sup = [ds for ds in DATASETS if M(ds, "BPSO", "acc_moy") >= M(ds, "AG", "acc_moy")]
if ag_sup:
    v4 = "infirmée" if len(ag_sup) == len(DATASETS) else "partiellement infirmée"
    H4 = (f"\\textbf{{H4 est {v4}}} : l'AG est significativement plus précis que le BPSO sur {liste(ag_sup)}.")
    LIT = (f"Nos résultats ne confirment donc pas entièrement cette tendance : à budget égal, l'AG est "
           f"significativement plus précis que le BPSO sur {liste(ag_sup)}.")
elif bpso_sup:
    v4 = "confirmée"
    H4 = (f"\\textbf{{H4 est confirmée}} : le BPSO n'est significativement moins précis que l'AG sur aucun jeu "
          f"de données, et il est significativement plus précis sur {liste(bpso_sup)}.")
    LIT = (f"Nos résultats vont dans le même sens : à budget égal, le BPSO est significativement plus précis "
           f"que l'AG sur {liste(bpso_sup)}, et jamais significativement moins précis.")
else:
    ag_moy_sup = [ds for ds in DATASETS if M(ds, "AG", "acc_moy") > M(ds, "BPSO", "acc_moy")]
    if len(ag_moy_sup) > len(DATASETS) / 2:
        v4 = "non confirmée"
        H4 = (f"\\textbf{{H4 n'est pas confirmée}} : aucune différence de précision entre AG et BPSO n'est "
              f"significative, et la précision moyenne de l'AG est supérieure à celle du BPSO sur "
              f"{liste(ag_moy_sup)}. Nos données ne permettent ni de confirmer l'avantage du BPSO, ni de "
              f"conclure à un avantage de l'AG.")
        LIT = ("Nos résultats ne reproduisent pas cette tendance : à budget égal et sur trois jeux de données, "
               "aucune différence de précision entre AG et BPSO n'est significative, et les moyennes favorisent "
               "plutôt l'AG.")
    else:
        v4 = "non infirmée"
        H4 = (f"\\textbf{{H4 n'est pas infirmée}} : la précision moyenne du BPSO est supérieure ou égale à celle "
              f"de l'AG sur {len(bpso_moy_sup)} des {len(DATASETS)} jeux, mais aucune différence n'est "
              f"significative ; l'avantage du BPSO n'est donc pas démontré.")
        LIT = ("Nos résultats nuancent cette tendance : à budget égal et sur trois jeux de données, aucune "
               "différence de précision entre AG et BPSO n'est statistiquement significative.")
out.append(macro("DiscHypotheses", "\n\n".join([H1, H2, H3, H4])))
out.append(macro("DiscLitterature",
    "Babaoğlu et al.~\\cite{babaoglu2010} et Rafdi et al.~\\cite{rafdi2021} rapportent un avantage du BPSO "
    "sur l'AG, chacun sur un seul jeu de données. " + LIT))

# écart cv / test
gaps = {}
for m in ["AG", "BPSO"]:
    g = []
    for ds in DATASETS:
        for r in range(R):
            row = res[(res.dataset == ds) & (res.methode == m) & (res.rep == r)].iloc[0]
            f_fin = curves[(ds, m, r)][-1]
            acc_cv = 1 - (f_fin - (1 - ALPHA) * row.nsel / NV[ds]) / ALPHA
            g.append(acc_cv - row.acc)
    gaps[m] = 100 * np.mean(g)
def phrase_gap(m, nm):
    v = gaps[m]
    return (f"pour {nm}, la précision estimée en validation croisée sur l'apprentissage "
            + (f"dépasse la précision mesurée sur le test de {num(v)}~point{'s' if abs(v) >= 2 else ''} en moyenne"
               if v >= 0 else f"est inférieure à la précision mesurée sur le test de {num(-v)}~point{'s' if abs(v) >= 2 else ''} en moyenne"))
out.append(macro("DiscEcart",
    "Sur l'ensemble des répétitions, " + phrase_gap("AG", "l'AG") + " ; " + phrase_gap("BPSO", "le BPSO") + ". "
    + ("Cet écart positif montre que la fitness optimisée est une estimation optimiste : à force d'évaluer des "
       "centaines de sous-ensembles sur les mêmes plis, la recherche finit par exploiter des particularités de "
       "l'ensemble d'apprentissage. C'est précisément pour mesurer ce phénomène que la précision est rapportée "
       "sur un ensemble de test jamais utilisé pendant la sélection \\cite{ambroise2002,kohavi1997}."
       if max(gaps.values()) > 0 else
       "La fitness optimisée ne surestime donc pas la performance réelle dans nos expériences, ce qui suggère "
       "un faible sur-apprentissage du critère de sélection.")))

# réponse à la question
ag_mieux_f = [ds for ds in DATASETS if M(ds, "AG", "acc_moy") > M(ds, "Filtre", "acc_moy") and P(ds, "p_acc_AG_Filtre") < 0.05]
ag_moins_f = [ds for ds in DATASETS if M(ds, "AG", "acc_moy") < M(ds, "Filtre", "acc_moy") and P(ds, "p_acc_AG_Filtre") < 0.05]
r_txt = f"{num(min(rA.values()), 0)} à {num(max(rA.values()), 0)} fois"
if ag_mieux_f:
    REP = (f"L'AG justifie son coût de calcul sur {liste(ag_mieux_f)}, où il améliore significativement la "
           f"précision du filtre, au prix d'un temps de sélection {r_txt} plus long.")
    REP_EN = (f"The GA justifies its computational cost on {liste(ag_mieux_f).replace(' et ', ' and ')}, where it "
              f"significantly improves on the filter's accuracy, at the price of a selection time "
              f"{num(min(rA.values()), 0).replace('{,}', '.')} to {num(max(rA.values()), 0).replace('{,}', '.')} times longer.")
else:
    REP = (f"Dans nos conditions expérimentales, l'AG n'améliore significativement la précision du filtre sur "
           f"aucun jeu de données, alors que sa sélection est {r_txt} plus lente : du point de vue de la "
           f"précision, son coût de calcul n'est pas justifié.")
    REP_EN = (f"Under our experimental conditions, the GA does not significantly improve on the filter's accuracy "
              f"on any dataset, while being {num(min(rA.values()), 0).replace('{,}', '.')} to "
              f"{num(max(rA.values()), 0).replace('{,}', '.')} times slower: in terms of accuracy, its "
              f"computational cost is not justified.")
if ag_moins_f:
    REP += f" Il est même significativement moins précis que le filtre sur {liste(ag_moins_f)}."
    REP_EN += f" It is even significantly less accurate than the filter on {liste(ag_moins_f).replace(' et ', ' and ')}."
out.append(macro("ReponseQuestion", REP))

d = DS_MAX
a = {m: M(d, m, "acc_moy") for m in METHODES}
n = {m: M(d, m, "nsel_moy") for m in METHODES}
out.append(macro("ResumeChiffres",
    f"Sur {d} ({NV[d]} variables), l'AG atteint {pct(a['AG'])}~\\% de précision avec {num(n['AG'])} variables "
    f"en moyenne, le BPSO {pct(a['BPSO'])}~\\% avec {num(n['BPSO'])} variables et le filtre {pct(a['Filtre'])}~\\% "
    f"avec {num(n['Filtre'])} variables, contre {pct(a['Toutes'])}~\\% avec toutes les variables ; la sélection "
    f"par AG est {num(rA[d], 0)} fois plus lente que le filtre. " + REP))
en = lambda x, k=1: f"{100 * x:.{k}f}"
out.append(macro("AbstractNumbers",
    f"On {d} ({NV[d]} features), the GA reaches {en(a['AG'])}\\% accuracy with {n['AG']:.1f} features on "
    f"average, BPSO {en(a['BPSO'])}\\% with {n['BPSO']:.1f} features and the filter {en(a['Filtre'])}\\% with "
    f"{n['Filtre']:.1f} features, versus {en(a['Toutes'])}\\% with all features; GA selection is "
    f"{rA[d]:.0f} times slower than the filter. " + REP_EN))

out.append(macro("ConclusionChapitre",
    f"Les résultats établissent les points suivants : H1 est {v1}, H2 est {v2}, H3 est {v3} et H4 est {v4}. " + REP))

# environnement
cpu = (platform.processor() or platform.machine()).replace("_", "\\_")
out.append(macro("Environnement",
    f"Les expériences ont été réalisées sous Python {platform.python_version()} avec NumPy {np.__version__} "
    f"et scikit-learn {sklearn.__version__} (Google Colab, processeur {cpu}, {os.cpu_count()} cœurs). "
    f"La durée totale des expériences est d'environ {max(1, round((time.perf_counter() - t_global) / 60))}~min."))

with open("resultats_auto.tex", "w", encoding="utf-8") as f:
    f.write("".join(out))
print("".join(out))